In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("sources", "yellow,green,zones")

In [0]:
import os, sys, yaml

catalog = dbutils.widgets.get("catalog")
sources = [s.strip() for s in dbutils.widgets.get("sources").split(",") if s.strip()]

src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from nyctaxi.bronze import read_stream, write_stream
from pyspark.sql import functions as F

with open("../config/sources.yml") as f:
  config = yaml.safe_load(f)

unknown = [s for s in sources if s not in config["sources"]]
if unknown:
    raise ValueError(f"This source doesn't appear in sources.yml: {unknown}")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.bronze.checkpoints")

In [0]:
for source in sources:
    cfg = config["sources"][source]
    
    glob_filter = None
    if cfg["format"] == "parquet":
        glob_filter = f"{source}_tripdata_2025-0[1-2].parquet" #just for testing, adjust it when pushing on azure

    source_dir = f"/Volumes/{catalog}/landing/raw_files/{source}/"
    checkpoint_dir = f"/Volumes/{catalog}/bronze/checkpoints/{source}"
    table = f"{catalog}.bronze.{cfg['bronze_table']}"

    df = read_stream(spark, source_dir, checkpoint_dir, cfg["format"], glob_filter)
    n = write_stream(df, checkpoint_dir, table)
    print(f"Read {n:,} records from {source_dir} to {table}")

    #["numInputRows"] on DB free edition returns None so terminal doesn't print real processed rows

      